In [8]:
# pip uninstall -y datasets huggingface_hub transformers

In [9]:
# %pip install datasets==4.4.1
# %pip install huggingface_hub==0.34.4
# %pip install transformers==4.56.2

In [10]:
import datasets
import huggingface_hub
import transformers

print(datasets.__version__)
print(huggingface_hub.__version__)
print(transformers.__version__)

4.4.1
0.34.4
4.56.2


# **task 2 : Text Summarization**

Summarization is a classic sequence-to-sequence (seq2seq) task with an input text and a target text.

In this part we will build our own encoder-decoder model to condense dialogues between several people into a crisp summary.

## The CNN/DailyMail Dataset

In [11]:
# from datasets import load_dataset
# dataset = load_dataset("cnn_dailymail", version="3.0.0")
# print(f"Features: {dataset['train'].column_names}")

from datasets import load_dataset

dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

print(f"[-] Dataset loaded successfully!")
print(f"Features: {dataset['train'].column_names}")


[-] Dataset loaded successfully!
Features: ['article', 'highlights', 'id']


The dataset has three columns: article, which contains the news articles, highlights with the summaries, and id to uniquely identify each article.

In [12]:
sample = dataset["train"][1]
print(f"""
Article (excerpt of 500 characters, total length: {len(sample["article"])}):
""")
print(sample["article"][:500])
print(f'\nSummary (length: {len(sample["highlights"])}):')
print(sample["highlights"])



Article (excerpt of 500 characters, total length: 4051):

Editor's note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events. Here, Soledad O'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial. MIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor." Here, inmates with the most s

Summary (length: 281):
Mentally ill inmates in Miami are housed on the "forgotten floor"
Judge Steven Leifman says most are there as a result of "avoidable felonies"
While CNN tours facility, patient shouts: "I am the son of the president"
Leifman says the system is unjust and he's fighting for change .


## Text Summarization Pipelines

we will restrict the input text to 2,000 characters to have the same input for all models and thus make the outputs more comparable

In [13]:
sample_text = dataset["train"][1]["article"][:2000]
# We'll collect the generated summaries of each model in a dictionary
summaries = {}

A convention in summarization is to separate the summary sentences by a newline. We could add a newline token after each full stop, but this simple
heuristic would fail for strings like “U.S.” or “U.N.” The Natural Language Toolkit (NLTK) package includes a more sophisticated algorithm that can
differentiate the end of a sentence from punctuation that occurs in abbreviations:

In [14]:
import nltk
from nltk.tokenize import sent_tokenize

# Download the packages required for sentence tokenization
# according to the latest NLTK updates
nltk.download("punkt")
nltk.download("punkt_tab")

# Test sentence to verify that abbreviations containing periods
# are handled correctly during sentence segmentation
string = "The U.S. are a country. The U.N. is an organization."

# Perform sentence tokenization and display the result
tokenized_output = sent_tokenize(string)
print("[-] Sentence tokenization structural check:")
print(tokenized_output)

[-] Sentence tokenization structural check:
['The U.S. are a country.', 'The U.N. is an organization.']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Summarization Baseline


A common baseline for summarizing news articles is to simply take the first three sentences of the article. With NLTK’s sentence tokenizer, we can easily implement such a baseline:


In [15]:
def three_sentence_summary(text):
  return "\n".join(sent_tokenize(text)[:3])


summaries["baseline"] = three_sentence_summary(sample_text)

### GPT-2

In this task, we use the GPT-2 model for text summarization. GPT-2 was originally designed for text generation, but it has shown the ability to produce concise summaries when provided with appropriate prompts. Its strong language modeling capabilities enable it to capture the main ideas of a document and generate coherent and fluent summaries without requiring a dedicated summarization architecture.

In [16]:
from transformers import pipeline, set_seed

set_seed(42)
pipe = pipeline("text-generation", model="gpt2-xl")

gpt2_query = sample_text + "\nTL;DR:\n"
pipe_out = pipe(gpt2_query, max_length=512, clean_up_tokenization_spaces=True)
summaries["gpt2"] = "\n".join(
 sent_tokenize(pipe_out[0]["generated_text"][len(gpt2_query) :]))


Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Technical Mechanism:**
We leverage **GPT-2 XL** to perform conditional text continuation rather than using a traditional Seq2Seq structure. By appending the prefix `\nTL;DR:\n` to the text, we trigger the model's pre-trained distribution to generate a concise summary based on internet-standard formatting patterns.

**Pipeline & Data Slicing:**
* **Sequence Isolation:** Since the model outputs both the prompt and the generated text, we apply an array slice `[len(gpt2_query):]` to isolate the new summary tokens.
* **Formatting Alignment:** The output is passed through `sent_tokenize()` and joined with newlines (`\n`) to maintain consistent sentence-level formatting across all evaluation baselines.

### T5

For this task, we also employ the T5 model for text summarization. T5 adopts a unified text-to-text framework, where all NLP tasks are formulated as text generation problems. Since it was pretrained on both unsupervised and supervised tasks, including summarization, it can generate summaries directly without additional fine-tuning. Its versatility and ability to handle multiple tasks using the same architecture make it an effective choice for automatic summarization.


In [17]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Explicitly load the tokenizer and the encoder-decoder interface for T5
t5_ckpt = "t5-large"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_ckpt)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_ckpt).to(
    device if "device" in locals() else "cpu"
)

# T5 requires a task-specific prefix to indicate summarization
t5_query = "summarize: " + sample_text

t5_inputs = t5_tokenizer(
    t5_query,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
).to(t5_model.device)

# Generate the summary independently of the input length
print("[-] Generating T5 summary with pure Seq2Seq architecture...")

t5_ids = t5_model.generate(
    t5_inputs["input_ids"],
    num_beams=4,
    max_new_tokens=100,
    min_new_tokens=30
)

# Decode the generated tokens and split the summary into sentences
summaries["t5"] = "\n".join(
    sent_tokenize(
        t5_tokenizer.decode(t5_ids[0], skip_special_tokens=True)
    )
)

print("[-] T5 Summary generated successfully!")

[-] Generating T5 summary with pure Seq2Seq architecture...
[-] T5 Summary generated successfully!


### BART


BART also uses an encoder-decoder architecture and is trained to reconstruct corrupted inputs. It combines the pretraining schemes of BERT and GPT-2.

In [18]:
# Explicitly load the tokenizer and the encoder-decoder interface for BART
bart_ckpt = "facebook/bart-large-cnn"
bart_tokenizer = AutoTokenizer.from_pretrained(bart_ckpt)

bart_model = AutoModelForSeq2SeqLM.from_pretrained(bart_ckpt).to(
    device if "device" in locals() else "cpu"
)

# Tokenize the input text and move it to the same device as the model
bart_inputs = bart_tokenizer(
    sample_text,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
).to(bart_model.device)

# Generate a summary using the pretrained BART model
print("[-] Generating BART summary with pure Seq2Seq architecture...")

bart_ids = bart_model.generate(
    bart_inputs["input_ids"],
    num_beams=4,
    max_new_tokens=100,
    min_new_tokens=30
)

# Decode the generated tokens and split the summary into sentences
summaries["bart"] = "\n".join(
    sent_tokenize(
        bart_tokenizer.decode(
            bart_ids[0],
            skip_special_tokens=True
        )
    )
)

print("[-] BART Summary generated successfully!")

[-] Generating BART summary with pure Seq2Seq architecture...
[-] BART Summary generated successfully!


### PEGASUS

Like BART, PEGASUS is an encoder-decoder transformer. its pretraining objective is to predict masked sentences in
multisentence texts.

In [19]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_ckpt = "google/pegasus-cnn_dailymail"

# 1. Explicitly load the tokenizer and the model
# as a complete Seq2Seq (Encoder-Decoder) architecture
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt)

# 2. Prepare the inputs and pass them to the appropriate device (GPU/CPU)
inputs = tokenizer(
    sample_text,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)

# 3. Generate the summary using the modern and reliable
# max_new_tokens parameter to avoid dimension-related issues
print("[-] Generating PEGASUS summary with pure Seq2Seq architecture...")

summary_ids = model.generate(
    inputs["input_ids"],
    num_beams=8,
    length_penalty=0.8,
    max_new_tokens=128,   # Controls the generated summary length independently
    min_new_tokens=30
)

# 4. Decode the generated tokens into clean text
pegasus_summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

# 5. Clean and format the output according to the PEGASUS
# strategy described in the book
summaries["pegasus"] = (
    pegasus_summary.strip()
    .replace(" .", ".\n")
)

print("[-] PEGASUS Summary generated successfully!")
print(summaries["pegasus"])

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[-] Generating PEGASUS summary with pure Seq2Seq architecture...
[-] PEGASUS Summary generated successfully!
Mentally ill inmates in Miami are housed on the "forgotten floor"<n>The ninth floor is where they're held until they're ready to appear in court.
<n>Most often, they face drug charges or charges of assaulting an officer.
<n>They end up on the ninth floor severely mentally disturbed.



## Comparing Different Summaries

In [20]:
print("GROUND TRUTH")
print(dataset["train"][1]["highlights"])
print("")
for model_name in summaries:
 print(model_name.upper())
 print(summaries[model_name])
 print("")


GROUND TRUTH
Mentally ill inmates in Miami are housed on the "forgotten floor"
Judge Steven Leifman says most are there as a result of "avoidable felonies"
While CNN tours facility, patient shouts: "I am the son of the president"
Leifman says the system is unjust and he's fighting for change .

BASELINE
Editor's note: In our Behind the Scenes series, CNN correspondents share their experiences in covering news and analyze the stories behind the events.
Here, Soledad O'Brien takes users inside a jail where many of the inmates are mentally ill. An inmate housed on the "forgotten floor," where many mentally ill inmates are housed in Miami before trial.
MIAMI, Florida (CNN) -- The ninth floor of the Miami-Dade pretrial detention facility is dubbed the "forgotten floor."

GPT2
The mentally ill aren't being treated properly and are being housed in an environment that is not conducive to their care.
The inmates are living on the ninth floor of the jail, a place where there are no beds and no p

* **BART & PEGASUS:** Demonstrate the highest semantic alignment with the `GROUND TRUTH`. **BART** captures key entities (`Judge Steven Leifman`) with exceptional accuracy, while **PEGASUS** matches the human summary structure closely but retains its raw newline token (`<n>`).
* **T5:** Yields a coherent abstractive summary that captures the core dilemma, though it features lower factual density compared to BART.
* **GPT-2:** Fails to maintain logical constraints, falling into an **autoregressive repetition loop** and factual hallucination. This confirms that models without explicit task-specific fine-tuning suffer from semantic decay.

## Measuring the Quality of Generated Text

Two of the most common metrics used to evaluate generated text are BLEU and ROUGE.

### BLEU Score:

* **The Core Idea:** It measures **Precision** by asking: *"Out of all the words generated by the model, how many are actually present in the human reference?"* It is designed to prevent the model from generating incorrect or irrelevant words.

In [21]:
!pip install -q evaluate sacrebleu
import evaluate

bleu_metric = evaluate.load("sacrebleu")

#### Empirical Verification of BLEU Core Mechanics:




In [22]:
import pandas as pd
import numpy as np
bleu_metric.add(
 prediction="the the the the the the", reference=["the cat is on the mat"])
results = bleu_metric.compute(smooth_method="floor", smooth_value=0)
results["precisions"] = [np.round(p, 2) for p in results["precisions"]]
pd.DataFrame.from_dict(results, orient="index", columns=["Value"])


,Value
score,0.0
counts,"[2, 0, 0, 0]"
totals,"[6, 5, 4, 3]"
precisions,"[33.33, 0.0, 0.0, 0.0]"
bp,1.0
sys_len,6
ref_len,6


### Example 1: Testing Word Repetition and N-gram Clipping
* **Result Interpretation:** The total score drops to **0.0** because the higher-order n-grams ($2$-gram to $4$-gram) yield zero matches, demonstrating how the clipped precision successfully penalizes repetitive, non-informative generations despite a baseline 1-gram precision of $33.33\%$.

In [23]:
bleu_metric.add(
 prediction="the cat is on mat", reference=["the cat is on the mat"])
results = bleu_metric.compute(smooth_method="floor", smooth_value=0)
results["precisions"] = [np.round(p, 2) for p in results["precisions"]]
pd.DataFrame.from_dict(results, orient="index", columns=["Value"])

,Value
score,57.893007
counts,"[5, 3, 2, 1]"
totals,"[5, 4, 3, 2]"
precisions,"[100.0, 75.0, 66.67, 50.0]"
bp,0.818731
sys_len,5
ref_len,6


### Example 2: Testing Token Omission and Brevity Penalty
* **Result Interpretation:** Although individual word precision is exceptionally high ($100\%$ for 1-grams), the overall score is compressed to **57.89** due to the activation of the brevity penalty ($bp = 0.81$), which penalizes the model for omitting necessary contextual tokens and producing a shorter sequence.

Integrating SacreBLEU into the Summary Baseline :

In [24]:
import pandas as pd

bleu_records = {}

reference = dataset["train"][1]["highlights"]

for model_name in summaries:
    # The SacreBLEU metric expects predictions as a list
    # and references as a nested list
    bleu_score = bleu_metric.compute(
        predictions=[summaries[model_name]],
        references=[[reference]]
    )

    # Extract the overall numerical score and round it
    # to two decimal places
    bleu_records[model_name] = round(
        bleu_score["score"],
        2
    )

# Automatically build the DataFrame from the computed results
df_bleu = pd.DataFrame.from_dict(
    bleu_records,
    orient="index",
    columns=["BLEU-4"]
)

df_bleu

,BLEU-4
baseline,8.38
gpt2,0.76
t5,6.00
bart,21.09
pegasus,20.52


The **BLEU** metric was originally designed and optimized for **Machine Translation** tasks, where strict **Precision** is the critical factor to ensure the model does not generate hallucinated or irrelevant tokens.

### ROUGE Score:

* **The Core Idea:** It measures **Recall** by asking: *"Out of all the important words in the human reference, how many did the model manage to capture?"* It is designed to ensure the model does not forget the main ideas of the text.

Key Differences Between ROUGE Variants :

* **ROUGE-1:** Measures the overlap of **single words** (1-grams) between the generated summary and the human reference to check basic vocabulary matching.
* **ROUGE-2:** Measures the overlap of **two-word pairs** (2-grams) to check the fluency and consecutive word order of the generation.
* **ROUGE-L:** Measures the **Longest Common Substring (LCS)** at the individual sentence level, recognizing flexible paraphrasing even if words are separated.
* **ROUGE-Lsum:** Measures the **Longest Common Substring** across the **entire summary layout** as a single block, ignoring sentence breaks (the official benchmark for CNN/DailyMail).

In [25]:
!pip install rouge_score
import evaluate

rouge_metric = evaluate.load("rouge")

In [26]:
reference = dataset["train"][1]["highlights"]
records = []
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

for model_name in summaries:

 rouge_metric.add(prediction=summaries[model_name], reference=reference)
 score = rouge_metric.compute()

 rouge_dict = {rn: score[rn] for rn in rouge_names}
 records.append(rouge_dict)

df_results = pd.DataFrame.from_records(records, index=summaries.keys())
df_results

,rouge1,rouge2,rougeL,rougeLsum
baseline,0.365079,0.145161,0.206349,0.285714
gpt2,0.159420,0.036496,0.130435,0.159420
t5,0.357895,0.107527,0.252632,0.357895
bart,0.444444,0.226804,0.323232,0.404040
pegasus,0.316832,0.202020,0.277228,0.316832


* **BART:** Achieves the highest scores overall. It leads with a **ROUGE-1 of 0.444** and **ROUGE-Lsum of 0.404**. This proves its fine-tuning captures key concepts perfectly.
* **PEGASUS:** Shows strong sequence continuity. It scores a competitive **0.202 on ROUGE-2**, maintaining consecutive word order effectively.
* **T5:** Delivers a solid baseline performance. However, its abstractive nature results in lower factual density than BART.
* **GPT-2:** Displays the lowest scores across the entire matrix. This drop is caused by the repetition loops and factual hallucinations.

## Evaluating PEGASUS on the CNN/DailyMail Dataset





In [27]:
def evaluate_summaries_baseline(dataset, metric,
 column_text="article",
 column_summary="highlights"):
 summaries = [three_sentence_summary(text) for text in dataset[column_text]]
 metric.add_batch(predictions=summaries,
 references=dataset[column_summary])
 score = metric.compute()
 return score

Establishing the Heuristic Baseline Floor :

To reduce computational cost and execution time, the evaluation was performed on a subset of 1,000 test samples instead of the entire dataset while still providing reliable performance estimates.


In [28]:
test_sampled = dataset["test"].shuffle(seed=42).select(range(1000))
score = evaluate_summaries_baseline(test_sampled, rouge_metric)
rouge_dict = {rn: score[rn] for rn in rouge_names}
df_baseline_test = pd.DataFrame.from_dict(rouge_dict, orient="index", columns=["baseline"]).T
df_baseline_test

,rouge1,rouge2,rougeL,rougeLsum
baseline,0.389276,0.171296,0.245061,0.354239


Now let’s implement the same evaluation function for evaluating the PEGASUS model:

In [29]:
from tqdm import tqdm
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

def chunks(list_of_elements, batch_size):
 """Yield successive batch-sized chunks from list_of_elements."""
 for i in range(0, len(list_of_elements), batch_size):
  yield list_of_elements[i : i + batch_size]

def evaluate_summaries_pegasus(dataset, metric, model, tokenizer,
                batch_size=16, device=device,
                column_text="article",
                column_summary="highlights"):

 article_batches = list(chunks(dataset[column_text], batch_size))
 target_batches = list(chunks(dataset[column_summary], batch_size))

 for article_batch, target_batch in tqdm(
  zip(article_batches, target_batches), total=len(article_batches)):
    inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                    padding="max_length", return_tensors="pt")
    summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                    attention_mask=inputs["attention_mask"].to(device),
                    length_penalty=0.8, num_beams=8, max_length=128)
    decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                    clean_up_tokenization_spaces=True)
              for s in summaries]
    decoded_summaries = [d.replace("<n>", " ") for d in decoded_summaries]
    metric.add_batch(predictions=decoded_summaries, references=target_batch)
 score = metric.compute()
 return score


* **`chunks()`**: Splits the dataset into smaller batches to facilitate processing and reduce memory consumption during evaluation.

* **`evaluate_summaries_pegasus()`**: Generates summaries using the PEGASUS model in batches and compares them with the reference summaries to compute evaluation metrics such as ROUGE.



In [ ]:
import gc
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Clear memory
gc.collect()
torch.cuda.empty_cache()

# Delete any previous models if they exist
for var in ["model", "tokenizer", "t5_model", "bart_model", "pipe"]:
    try:
        del globals()[var]
    except:
        pass

gc.collect()
torch.cuda.empty_cache()

# Load PEGASUS
model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

# Run evaluation with lighter memory settings
print("[-] Running PEGASUS evaluation...")

small_test = test_sampled.select(range(100))

score = evaluate_summaries_pegasus(
    small_test,
    rouge_metric,
    model,
    tokenizer,
    batch_size=1   # instead of 8
)

# Extract results
rouge_dict = {rn: score[rn] for rn in rouge_names}

# Display as DataFrame
df_pegasus_test = pd.DataFrame(
    rouge_dict,
    index=["pegasus"]
)

df_pegasus_test

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[-] Running PEGASUS evaluation...


 81%|████████  | 81/100 [06:31<01:13,  3.89s/it]

## Training a Summarization Model

In [ ]:
from datasets import load_dataset


dataset_samsum = load_dataset("knkarthick/samsum")


In [ ]:
# dataset_samsum = load_dataset("samsum_dataset")
split_lengths = [len(dataset_samsum[split])for split in dataset_samsum]
print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue:")
print(dataset_samsum["test"][0]["dialogue"])
print("\nSummary:")
print(dataset_samsum["test"][0]["summary"])

### Evaluating PEGASUS on SAMSum

In [ ]:
pipe_out = pipe(dataset_samsum["test"][0]["dialogue"])
print("Summary:")
print(pipe_out[0]["summary_text"].replace(" .<n>", ".\n"))


In [ ]:
score = evaluate_summaries_pegasus(dataset_samsum["test"], rouge_metric, model,
 tokenizer, column_text="dialogue",
 column_summary="summary", batch_size=8)
rouge_dict = dict((rn, score[rn].mid.fmeasure) for rn in rouge_names)
pd.DataFrame(rouge_dict, index=["pegasus"])

### Fine-Tuning PEGASUS

In [ ]:
import matplotlib.pyplot as plt

d_len = [len(tokenizer.encode(s)) for s in dataset_samsum["train"]["dialogue"]]
s_len = [len(tokenizer.encode(s)) for s in dataset_samsum["train"]["summary"]]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
axes[0].hist(d_len, bins=20, color="C0", edgecolor="C0")
axes[0].set_title("Dialogue Token Length")
axes[0].set_xlabel("Length")
axes[0].set_ylabel("Count")
axes[1].hist(s_len, bins=20, color="C0", edgecolor="C0")
axes[1].set_title("Summary Token Length")
axes[1].set_xlabel("Length")
plt.tight_layout()
plt.show()

In [ ]:
def convert_examples_to_features(example_batch):
 input_encodings = tokenizer(example_batch["dialogue"], max_length=1024,
 truncation=True)
 with tokenizer.as_target_tokenizer():
 target_encodings = tokenizer(example_batch["summary"], max_length=128,
 truncation=True)
 return {"input_ids": input_encodings["input_ids"],
 "attention_mask": input_encodings["attention_mask"],
 "labels": target_encodings["input_ids"]}
dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features,
 batched=True)
columns = ["input_ids", "labels", "attention_mask"]
dataset_samsum_pt.set_format(type="torch", columns=columns)


In [ ]:
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


In [ ]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
 output_dir='pegasus-samsum', num_train_epochs=1, warmup_steps=500,
 per_device_train_batch_size=1, per_device_eval_batch_size=1,
 weight_decay=0.01, logging_steps=10, push_to_hub=True,
 evaluation_strategy='steps', eval_steps=500, save_steps=1e6,
 gradient_accumulation_steps=16)


### Generating Dialogue Summaries

In [ ]:
gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}
sample_text = dataset_samsum["test"][0]["dialogue"]
reference = dataset_samsum["test"][0]["summary"]
pipe = pipeline("summarization", model="transformersbook/pegasus-samsum")
print("Dialogue:")
print(sample_text)
print("\nReference Summary:")
print(reference)
print("\nModel Summary:")
print(pipe(sample_text, **gen_kwargs)[0]["summary_text"])


In [ ]:
custom_dialogue = """\
Thom: Hi guys, have you heard of transformers?
Lewis: Yes, I used them recently!
Leandro: Indeed, there is a great library by Hugging Face.
Thom: I know, I helped build it ;)
Lewis: Cool, maybe we should write a book about it. What do you think?
Leandro: Great idea, how hard can it be?!
Thom: I am in!
Lewis: Awesome, let's do it together!
"""
print(pipe(custom_dialogue, **gen_kwargs)[0]["summary_text"])